In [ ]:
import cv2
from ultralytics import YOLO
from pathlib import Path
import subprocess
import ffmpeg
import cv2
import numpy as np
import yt_dlp

In [4]:
model = YOLO("/Users/magewade/Desktop/ML/puppies_detection/weights/best.pt")

In [16]:
conf = 0.3
iou = 0.5

Тут можно задетектить видео прям со стрима

In [21]:
# 🎯 Получаем прямую ссылку на видеопоток + размер кадра
def get_stream_info(youtube_url):
    ydl_opts = {
        "quiet": True,
        "format": "best[ext=mp4]/best",
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(youtube_url, download=False)
        width = info.get("width", 480)
        height = info.get("height", 290)
        return info["url"], width, height


# 📺 Ссылка на YouTube
youtube_url = "https://www.youtube.com/watch?v=bYlEgU2tU5w"
stream_url, frame_width, frame_height = get_stream_info(youtube_url)

print(f"📺 Получен поток: {stream_url}")
print(f"📐 Размер кадра: {frame_width}x{frame_height}")

# 🧵 Настройка ffmpeg
ffmpeg_cmd = [
    "ffmpeg",
    "-i",
    stream_url,
    "-vf",
    f"scale={frame_width}:{frame_height}",  # убедимся, что размер фиксированный
    "-f",
    "image2pipe",
    "-pix_fmt",
    "bgr24",
    "-vcodec",
    "rawvideo",
    "-loglevel",
    "quiet",
    "-",
]
pipe = subprocess.Popen(ffmpeg_cmd, stdout=subprocess.PIPE)

frame_size = frame_width * frame_height * 3
frame_count = 0
skip_every = 1  # обрабатывать каждый 5-й кадр
results = None  # последние результаты YOLO

try:
    while True:
        raw_frame = pipe.stdout.read(frame_size)
        if not raw_frame:
            print("🚫 Поток завершился или прервался")
            break

        frame = np.frombuffer(raw_frame, dtype=np.uint8)
        if frame.size != frame_size:
            print("⚠️ Размер кадра не совпадает, пропуск...")
            continue

        frame = frame.reshape((frame_height, frame_width, 3))

        frame_count += 1
        if frame_count % skip_every == 0:
            results = model.track(
                source=frame,
                persist=True,
                tracker="puppy_tracker.yaml",
                verbose=False,
            )

        if results:
            annotated = results[0].plot()
        else:
            annotated = frame

        cv2.imshow("YOLO Stream", annotated)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

except KeyboardInterrupt:
    print("⛔ Остановлено пользователем")

finally:
    pipe.terminate()
    cv2.destroyAllWindows()

📺 Получен поток: https://manifest.googlevideo.com/api/manifest/hls_playlist/expire/1745246227/ei/swMGaPvqLdiH0u8Pt7rWgQw/ip/158.181.215.235/id/bYlEgU2tU5w.1/itag/96/source/yt_live_broadcast/requiressl/yes/ratebypass/yes/live/1/sgoap/gir%3Dyes%3Bitag%3D140/sgovp/gir%3Dyes%3Bitag%3D137/rqh/1/hls_chunk_host/rr3---sn-hxb5apox-4g0s.googlevideo.com/xpc/EgVo2aDSNQ%3D%3D/playlist_duration/30/manifest_duration/30/bui/AccgBcMmSqe0nUZOiiH17ELudyDGeOy428xTy9hwtor4oyGRMpqkXX7EyBuPHtaxgn1moaPVLmP-AIRd/spc/_S3wKgT8Yw3mvIafDsC_4vI8voYVcT3Bpz3IDn80A2QXaTGOOHSJDub2z6SY76HSZrb1IhM/vprv/1/playlist_type/DVR/initcwndbps/1748750/met/1745224629,/mh/uK/mm/44/mn/sn-hxb5apox-4g0s/ms/lva/mv/m/mvi/3/pl/20/rms/lva,lva/dover/11/pacing/0/keepalive/yes/fexp/51355912/mt/1745224110/sparams/expire,ei,ip,id,itag,source,requiressl,ratebypass,live,sgoap,sgovp,rqh,xpc,playlist_duration,manifest_duration,bui,spc,vprv,playlist_type/sig/AJfQdSswRQIgGLwiHET9o-dX23IuQ6WC3a-wvsC5rZMJjEQJzX9cPXMCIQCqUhx_yGzB-zoa3O6j4F8n8nAqCEJnOGVa

А тут поинференсить сохраненное видео

In [22]:
results = model.track(
    source="/Users/magewade/Desktop/ML/puppies_detection/video/infer/puppies_inference_7.mp4",
    tracker="puppy_tracker.yaml",
    show=True,
)


WARNING ⚠️ inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs



KeyboardInterrupt: 